# Possession run 02: explicitly accepted draft annotations

Private CPU experiment. Train on the supplied public five minutes plus the bounded
second-half draft intervals. Evaluate once on the bounded first-half draft intervals.
The calibration is not an additional new labelled window. Unmarked prefixes and
open tails remain unknown. Draft status is preserved: these are provisional scores,
not a certified or final-test benchmark. No validation-based parameter search.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working/possession_run02')
WORK.mkdir(parents=True,exist_ok=True)
roots=[p.parent for p in INPUT.rglob('requirements-learning.txt')
       if (p.parent/'tools/import_review_draft.py').is_file()]
assert len(roots)==1, f'Expected one current source bundle, found {roots}'
CODE=roots[0]
audits=list(INPUT.rglob('annotation_audit.json'))
assert len(audits)==1, f'Expected one supplied annotation audit, found {audits}'
LABELS=audits[0].parent
audit=json.loads(audits[0].read_text())
def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
for name in ('train','validation'):
    assert sha(LABELS/f'{name}.csv')==audit[f'{name}_labels_sha256']
features=list(INPUT.rglob('features.npz'))
expected_features='dc8c85c46bb381eaa8b1bb6deeb004f458165237bb00b7e89ba4fc4a756f805a'
features=[p for p in features if sha(p)==expected_features]
assert len(features)==1, 'Attach the previous private smoke notebook output with the frozen feature cache.'
FEATURES=features[0]
old_predictions=[p for p in INPUT.rglob('learned.csv') if p.parent.name=='public_smoke']
assert len(old_predictions)==1, 'Expected previous smoke predictions for a same-window comparison.'
OLD=old_predictions[0]
print('Evaluation: PROVISIONAL DRAFT HOLDOUT',flush=True)
print('Source:',CODE,'Annotations:',LABELS,'Cached features:',FEATURES,flush=True)
print('Training counts:',audit['training_counts'],'Validation counts:',audit['validation_counts'],flush=True)


Install the pinned experimental stack in /tmp so unrelated Kaggle packages and
saved output directories are untouched. Run the exporter/leakage/decoder tests.


In [ ]:
DEPS=Path('/tmp/barca_run02_python')
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
                '--target',str(DEPS),'-r',str(CODE/'requirements-learning.txt')],check=True)
env=dict(os.environ,PYTHONPATH=str(DEPS)+os.pathsep+str(CODE),PYTHONDONTWRITEBYTECODE='1')
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=CODE,env=env,check=True)
run('-m','unittest','discover','-s','tests','-v')


Recreate the label files from the original user JSON, with the same explicit
window assignment. Verify their hashes before fitting. Repeated same-label marks
do not create a possession; prefixes and open last segments remain X. The first-half
window's original test allocation becomes provisional validation and is recorded.


In [ ]:
REBUILT=WORK/'recreated_labels'
run('tools/import_review_draft.py','--plan',LABELS/'plan.json','--review',LABELS/'raw_review.json',
    '--train-window','train_h2_05','--validation-window','test_h1_01',
    '--include-public-training','--allow-drafts','--out',REBUILT)
for name in ('train','validation'):
    assert sha(REBUILT/f'{name}.csv')==sha(LABELS/f'{name}.csv')
print('Annotation files reproduced byte for byte.',flush=True)
RUN=WORK/'run'
run('-m','learned_possession','train','--features',FEATURES,
    '--train-labels',REBUILT/'train.csv','--validation-labels',REBUILT/'validation.csv',
    '--out',RUN,'--seed','0','--iterations','100','--threads','4','--bootstrap','1000',
    '--compare',CODE/'evaluation/results/possession_v6.csv',
    CODE/'evaluation/results/baseline_smooth50.csv',OLD)


In [ ]:
# Preserve the provenance in both the saved model and the score report. This
# changes metadata only; fitted classifiers and frozen decoder settings stay fixed.
run('-c',"""
from pathlib import Path
import json,joblib,sys
run=Path(sys.argv[1]);audit=json.loads(Path(sys.argv[2]).read_text())
metadata=dict(evaluation_kind=audit['evaluation_kind'],annotation_status=audit['annotation_status'],
              source_review_sha256=audit['source_review_sha256'])
bundle=joblib.load(run/'model.joblib');bundle['metadata'].update(metadata)
joblib.dump(bundle,run/'model.joblib')
report=json.loads((run/'report.json').read_text());report['training'].update(metadata)
report['annotation_audit']=audit
(run/'report.json').write_text(json.dumps(report,indent=2,sort_keys=True)+chr(10))
""",RUN,REBUILT/'annotation_audit.json')
run('-m','learned_possession','predict','--features',FEATURES,'--model',RUN/'model.joblib',
    '--out',RUN/'regenerated.csv','--threads','4')
assert (RUN/'learned.csv').read_bytes()==(RUN/'regenerated.csv').read_bytes()
protocol=dict(evaluation_kind=audit['evaluation_kind'],feature_cache_sha256=sha(FEATURES),
              annotation_audit=audit,prediction_sha256=sha(RUN/'learned.csv'),
              final_test_evaluated=False,parameter_search=False,
              code_sha256={name:sha(CODE/name) for name in [
                  'requirements-learning.txt','learned_possession/model.py',
                  'learned_possession/segment.py','learned_possession/features.py',
                  'learned_possession/__main__.py','tools/import_review_draft.py',
                  'evaluation/possession_metrics.py']})
(WORK/'protocol.json').write_text(json.dumps(protocol,indent=2)+chr(10))
report=json.loads((RUN/'report.json').read_text())
print('PROVISIONAL SCORES ON THE SAME DRAFT VALIDATION FRAMES',flush=True)
for name,score in report['scores'].items():
    print(name, 'spellF1',round(score['spell_f1_mean'],4),
          'macroF1',round(score['macro_f1'],4),'possessions',score['n_pred_spells'],
          '/',score['n_gt_spells'],flush=True)
print('Saved-model regeneration is byte-identical:',protocol['prediction_sha256'],flush=True)
